# Multi-Agent Workflow for Biomedical Literature Review

## 1. Setup & Data Access

Install dependencies, configure S3 client and E-utilities, search PMC for articles (3 test queries), fetch XMLs from S3, parse XML → structured JSON, and run quality checks.

### Install Dependencies

Run once to install all required packages.

In [ ]:
%pip install boto3 requests lxml chromadb sentence-transformers transformers torch pandas jupyter

### Imports

In [1]:
import requests
import time
import json
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from lxml import etree

### Step 1 — Article Discovery via PubMed E-utilities

Use NCBI's E-utilities API to search PubMed Central by topic and retrieve PMC IDs.

In [ ]:
def search_pmc(query: str, max_results: int = 30) -> list[str]:
    """Search PMC and return list of PMC IDs."""
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pmc",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    pmc_ids = data["esearchresult"]["idlist"]
    return [f"PMC{pid}" for pid in pmc_ids]

In [8]:
# Test queries from the plan
test_queries = [
    "mRNA vaccine adverse events pediatric",
    "Transformer-based models for protein folding",
    "Clinical trial outcomes for monoclonal antibodies in oncology"
]

# Search for all 3 queries, collect PMC IDs, deduplicate
all_pmc_ids = set()
for query in test_queries:
    print(f"Searching: {query}")
    pmc_ids = search_pmc(query, max_results=30)
    all_pmc_ids.update(pmc_ids)
    print(f"  Found {len(pmc_ids)} IDs")
    time.sleep(0.5)  # Rate limiting: 3 req/sec without API key

all_pmc_ids = list(all_pmc_ids)
print(f"\nTotal unique PMC IDs: {len(all_pmc_ids)}")

Searching: mRNA vaccine adverse events pediatric
  Found 30 IDs
Searching: Transformer-based models for protein folding
  Found 30 IDs
Searching: Clinical trial outcomes for monoclonal antibodies in oncology
  Found 30 IDs

Total unique PMC IDs: 90


In [ ]:
# test the first query (in slightly nicer form) to ensure we get results
import requests
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pmc",
    "term": "mRNA vaccine adverse events pediatric",
    "retmax": 30,
    "retmode": "json"
}
r = requests.get(url, params=params)
print(r.json()["esearchresult"]["count"])

10686


### Step 2 — Fetch XML Files from S3

Download full-text XML for each PMC ID from the public PubMed Central Open Access bucket.

In [ ]:

def fetch_xml_from_s3(pmc_id: str) -> str | None:
    """Download XML for a given PMC ID from S3, trying both old and new paths."""
    # New structure (for newer articles): PMC12345678.1/PMC12345678.1.xml
    new_key = f"{pmc_id}.1/{pmc_id}.1.xml"
    # Old structure (available until Aug 2026): oa_comm/xml/all/PMC12345678.xml
    old_key = f"oa_comm/xml/all/{pmc_id}.xml"
    
    for key in [new_key, old_key]:
        try:
            obj = s3.get_object(Bucket="pmc-oa-opendata", Key=key)
            return obj["Body"].read().decode("utf-8")
        except Exception:
            continue
    
    print(f"WARNING: {pmc_id} not found in oa_comm. Skipping.")
    return None


In [ ]:
# Fetch XML for all PMC IDs
raw_xmls = []
for i, pmc_id in enumerate(all_pmc_ids):
    xml = fetch_xml_from_s3(pmc_id)
    if xml is not None:
        raw_xmls.append(xml)
    if (i + 1) % 10 == 0:
        print(f"Fetched {i + 1}/{len(all_pmc_ids)}...")

print(f"\nSuccessfully fetched {len(raw_xmls)} XML files")

### Step 3 — Parse XML to Structured JSON

Parse each JATS XML file and extract structured fields.

In [10]:
def parse_article_xml(xml_string: str) -> dict | None:
    """Parse JATS XML and return structured article dict."""
    try:
        root = etree.fromstring(xml_string.encode("utf-8"))
    except etree.XMLSyntaxError:
        return None

    def get_text(element):
        """Recursively get all text from an element and its children."""
        return "".join(element.itertext()).strip() if element is not None else ""

    front = root.find(".//front")
    article_meta = front.find(".//article-meta") if front is not None else None

    if article_meta is None:
        return None

    # PMC ID
    pmc_id = ""
    for aid in article_meta.findall(".//article-id"):
        if aid.get("pub-id-type") == "pmc":
            pmc_id = aid.text or ""
            break

    # PMID
    pmid = ""
    for aid in article_meta.findall(".//article-id"):
        if aid.get("pub-id-type") == "pmid":
            pmid = aid.text or ""
            break

    # Title
    title_el = article_meta.find(".//article-title")
    title = get_text(title_el)

    # Abstract
    abstract_el = article_meta.find(".//abstract")
    abstract = get_text(abstract_el) if abstract_el is not None else ""

    # Authors
    authors = []
    for contrib in article_meta.findall(".//contrib[@contrib-type='author']"):
        surname = contrib.findtext(".//surname", default="")
        given = contrib.findtext(".//given-names", default="")
        if surname:
            authors.append(f"{given} {surname}".strip())

    # Journal
    journal = ""
    journal_el = front.find(".//journal-title") if front is not None else None
    if journal_el is not None:
        journal = get_text(journal_el)

    # Publication date
    pub_date = ""
    pub_date_el = article_meta.find(".//pub-date")
    if pub_date_el is not None:
        year = pub_date_el.findtext("year", default="")
        month = pub_date_el.findtext("month", default="")
        day = pub_date_el.findtext("day", default="")
        pub_date = f"{year}-{month.zfill(2)}-{day.zfill(2)}" if year else ""

    # Keywords
    keywords = []
    for kwd in article_meta.findall(".//kwd"):
        kw_text = get_text(kwd)
        if kw_text:
            keywords.append(kw_text)

    # Body text (for future RAG use)
    body_el = root.find(".//body")
    body_text = get_text(body_el) if body_el is not None else ""

    # Normalize PMC ID format (ensure PMC prefix)
    if pmc_id and not pmc_id.upper().startswith("PMC"):
        pmc_id = f"PMC{pmc_id}"

    return {
        "pmc_id": pmc_id,
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "authors": authors,
        "journal": journal,
        "pub_date": pub_date,
        "keywords": keywords,
        "body_text": body_text[:5000]
    }

In [11]:
# Parse all XMLs and filter out articles without abstracts
articles = [parse_article_xml(xml) for xml in raw_xmls if xml is not None]
articles = [a for a in articles if a is not None and a["abstract"]]

# Save to JSON
with open("articles.json", "w") as f:
    json.dump(articles, f, indent=2)

print(f"Saved {len(articles)} articles to articles.json")

Saved 2 articles to articles.json


### Quality Check

Print stats: total articles, with abstracts, with keywords.

In [12]:
total = len(articles)
with_abstracts = sum(1 for a in articles if a.get("abstract"))
with_keywords = sum(1 for a in articles if a.get("keywords"))

print("=== Quality Check ===")
print(f"Total articles:        {total}")
print(f"With abstracts:       {with_abstracts}")
print(f"With keywords:        {with_keywords}")
print(f"Without abstracts:    {total - with_abstracts} (excluded from index)")

=== Quality Check ===
Total articles:        2
With abstracts:       2
With keywords:        2
Without abstracts:    0 (excluded from index)
